Importing Needed Libraries - Routing to Proper Device

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image

print("torch:", torch.__version__)
device = "cuda"  # Change device in the runtime settings - to GPU T4
print("device:", device)

def set_seed(seed: int = 42):                                                                       # Here for reproducability
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)                                                                                        # Here for reproducability


# malco was here
# sarah was here

torch: 2.11.0+cpu
device: cuda


Importing From Kaggle API - Get Kaggle key first - Already Downloaded

In [ ]:
!export KAGGLE_API_TOKEN=KGAT_6e0db17c19cd9ec6f55621b971c1014d

!mkdir -p ~/.kaggle && echo KGAT_6e0db17c19cd9ec6f55621b971c1014d > ~/.kaggle/access_token && chmod 666 ~/.kaggle/access_token

!kaggle competitions list

!export KAGGLE_API_TOKEN=KGAT_d77bc5e53c781f271763950cf13abd1c

!mkdir -p ~/.kaggle && echo KGAT_d77bc5e53c781f271763950cf13abd1c > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

!kaggle competitions list

ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                               2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                       2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection             2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3          

In [ ]:
# Code that kaggle said to run to get access to the competition files
import kagglehub

# Download latest version
path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/ucsc-cse-144-spring-2026-final-project


Data Pipeline - Dataset and DataLoader - IN PROGRESS

In [ ]:

batch_size = 64  # For now, can change to 64?
num_workers = 0                                                                 # Here for reproducability
mean=[0.485, 0.456, 0.406]                                                      # ImageNet stats - that is what the model was trained on
std=[0.229, 0.224, 0.225]
val_p = 0.2 # portion of training data


class UnlabeledDataset(Dataset):
  def __init__(self, root, transform=None):
      self.root   = root
      self.transform = transform
      self.images    = sorted(os.listdir(root))

  def __len__(self):
      return len(self.images)

  def __getitem__(self, idx):
      img_path = os.path.join(self.root, self.images[idx])
      image    = Image.open(img_path).convert('RGB')
      if self.transform:
          image = self.transform(image)
      return image, self.images[idx]

train_path = path + '/train'
test_path = path + '/test'

train_tf = v2.Compose([
           v2.ToImage(),
           v2.ToDtype(torch.uint8, scale=True),
           v2.Resize((224, 224)),
           v2.RandomResizedCrop(size=(224, 224), antialias=True),   # Augmentation
           v2.RandomHorizontalFlip(p=0.5),                          # Augmentation
           v2.ToDtype(torch.float32, scale=True),
           v2.Normalize(mean, std)
])

test_tf = v2.Compose([
          v2.ToImage(),
          v2.Resize((224, 224)),
          v2.Normalize(mean, std)
])

train_set = datasets.ImageFolder(root=train_path, transform=train_tf)
val_set   = datasets.ImageFolder(root=train_path,  transform=test_tf)
test_set   = UnlabeledDataset(root=test_path,  transform=test_tf)

train_size = int((1-val_p) * len(train_set))
val_size   = len(train_set) - train_size

train_set, val_set = random_split(train_set, [train_size, val_size])

train_loader = DataLoader(dataset=train_set,
                          batch_size=64,
                          shuffle=True,
                          num_workers=0
)
val_loader =   DataLoader(dataset=val_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
)
test_loader =  DataLoader(dataset=test_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
)

print("train/val/test:", len(train_set), len(val_set), len(test_set))

train/val/test: 863 216 1036


Define the Model

In [ ]:
class MODEL(nn.Module):
  # CNN from my project
  def __init__(self, num_classes=10):
      super().__init__()
      # ========== YOUR CODE STARTS HERE ==========
      # Build a CNN with:
      # - Feature extraction: 3 convolutional blocks
      self.features = nn.Sequential(
          #   * First block: 1 input channel to 32 output channels, with BatchNorm and ReLU, then MaxPool
          nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
          nn.BatchNorm2d(num_features=32),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2, stride=2),                          # 28x28 → 14x14

          #   * Second block: 32 to 64 channels, with BatchNorm and ReLU, then MaxPool
          nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
          nn.BatchNorm2d(num_features=64),
          nn.ReLU(),
          nn.MaxPool2d(2, 2),                           # 14x14 → 7x7

          #   * Third block: 64 to 128 channels, with ReLU (no pooling)
          nn.Conv2d(64, 128, kernel_size=3, padding=1),
          nn.ReLU(),
      )   #   Use kernel_size=3 and padding=1 for all convolutions, pool_size=2 for pooling

      # - Classifier: Flatten, then fully connected layers
      #   * After two MaxPool layers, your feature map will be 7x7
      #   * First FC layer: 128*7*7 inputs to 256 outputs, with ReLU and Dropout(0.3)
      #   * Final FC layer: 256 to num_classes outputs

      self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(128 * 7 * 7, 256),
          nn.ReLU(),
          nn.Dropout(0.3),
          nn.Linear(256, num_classes)
      )
      # ========== YOUR CODE ENDS HERE ============

  def forward(self, x):
      # ========== YOUR CODE STARTS HERE ==========
      # Pass input through features, then classifier
      x = self.features(x)        # Represents the convolutional block (4d tensor)
      logits = self.classifier(x) # Represents the fully connected layer of the stack
      return logits
      # ========== YOUR CODE ENDS HERE ============

model = MODEL().to(device)
print(model)

num_params = sum(p.numel() for p in model.parameters())
print("Total params:", num_params)

Training Loop